# Whole-Brain Within-Source Searchlight Decoding — ERP (stimulus-averaged)

This notebook is the **within-source** counterpart to `searchlight_crossdecoding_vts_reworked.ipynb`.

- **Cross-source** = train on Natural, test on AI (and vice versa).
- **Within-source** (this notebook) = train and test on the **same** source: Natural→Natural and AI→AI.

The decoding follows the **ERP / stimulus-averaged** logic from your ROI notebook
(`decoding_multisub_parallel_avg_zscore_voxel_source`, function `decode_within_erp`):
single-trial betas are first **averaged within each stimulus identity**, giving one pattern per
unique image, and an SVM is trained/tested with RepeatedStratifiedKFold inside that source.

**Z-scoring.** Because this is within-source and ERP-averaged, normalization is done in the way
that fits this design and does not leak across the train/test split:
- Pattern normalization (z-score across voxels within each stimulus-averaged pattern) inside each
  searchlight sphere. This is per-sample, so it is identical whether a pattern lands in train or test.
- Voxel-wise standardization is then fit **on the training fold only** and applied to the test fold
  (no leakage), matching the inside-fold standardization style used elsewhere in your pipeline.

Source-level z-scoring is computed **per source separately** (Natural patterns standardized using
Natural statistics, AI using AI statistics), which is the correct choice for within-source decoding.

Structural choices (KD-tree searchlight grid, mask-column vs flat-index bookkeeping, sphere validity
checks, diagnostic counters, per-subject / mean / std / n-subject NIfTI outputs) are carried over
unchanged from the cross-source notebook.


In [1]:
import os, gc, warnings, pickle
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import scipy.io
import nibabel as nib
from nilearn.image import resample_img
from scipy.spatial import cKDTree
from scipy.stats import zscore
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

warnings.filterwarnings("ignore")

## Configuration

In [2]:
# ============================
# Paths
# ============================
beta_dir = r'N:\Experimental_Data\yujunchen\projects\AI_IAPS\GLM_singletrial\betas'
label_file = r'N:\Experimental_Data\yujunchen\projects\AI_IAPS\GLM_singletrial\beta_groups.csv'
mask_file = r'N:\Experimental_Data\yujunchen\projects\data\masks\MNI152_T1_2mm_brain_mask.nii.gz'
onset_base_dir = r'N:\Experimental_Data\yujunchen\projects\LAB_IAPS_AI\DataRecording'

output_dir = r'N:\Experimental_Data\yujunchen\projects\AI_IAPS\Decoding\searchlight_within_source_erp'
os.makedirs(output_dir, exist_ok=True)

# ============================
# Subjects / runs
# ============================
subs = ['Sub1', 'Sub2', 'Sub3', 'Sub4', 'Sub5', 'Sub6', 'Sub7', 'Sub8', 'Sub9',
        'Sub11', 'Sub12', 'Sub13', 'Sub14', 'Sub15', 'Sub16', 'Sub17', 'Sub18',
        'Sub19', 'Sub20', 'Sub21', 'Sub22', 'Sub23', 'Sub24', 'Sub25', 'Sub26',
        'Sub27', 'Sub28', 'Sub29', 'Sub30', 'Sub31']

runs = ['Run01','Run02','Run03','Run04','Run05','Run06','Run07','Run08','Run09','Run10']

# ============================
# Searchlight / decoding parameters
# ============================
RADIUS_MM = 5.0
MIN_VOXELS = 10

# ERP cross-validation (matches decode_within_erp in the ROI notebook).
ERP_N_REPEATS = 10
ERP_N_FOLDS = 5

# Inside-fold voxel standardization fit on training patterns only.
USE_TRIAL_ZSCORE_IN_FOLD = True

# Whole-brain searchlight is expensive. Start with 1-2 jobs and increase only if memory is safe.
N_JOBS = 10

# Debug options
DEBUG_FIRST_SUB_ONLY = False
MAX_CENTERS_DEBUG = None   # e.g., 2000 for testing; None = all centers

natural_cats = ['pleasant', 'neutral', 'unpleasant']
ai_cats = ['pleasantAI', 'neutralAI', 'unpleasantAI']
all_categories = natural_cats + ai_cats

# Within-source comparisons: train and test on the SAME source.
within_comparisons = [
    ('pleasant', 'neutral'),          # Natural
    ('pleasantAI', 'neutralAI'),      # AI
    ('unpleasant', 'neutral'),        # Natural
    ('unpleasantAI', 'neutralAI'),    # AI
]

comp_names = [f"{a}_vs_{b}" for a, b in within_comparisons]

print(f"Output directory: {output_dir}")
print(f"Comparisons: {comp_names}")

Output directory: N:\Experimental_Data\yujunchen\projects\AI_IAPS\Decoding\searchlight_within_source_erp
Comparisons: ['pleasant_vs_neutral', 'pleasantAI_vs_neutralAI', 'unpleasant_vs_neutral', 'unpleasantAI_vs_neutralAI']


## Build stimulus → category lookup

In [3]:
def load_stim_names_for_subject(sub, runs):
    sub_onset_dir = os.path.join(onset_base_dir, sub, 'LogFiles')
    stim_names = []

    for run_name in runs:
        fpath = os.path.join(sub_onset_dir, f'{run_name}.mat')
        if not os.path.exists(fpath):
            alt_name = 'Run' + str(int(run_name.replace('Run', '')))
            fpath = os.path.join(sub_onset_dir, f'{alt_name}.mat')

        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Cannot find onset file for {sub}, {run_name}: {fpath}")

        mat = scipy.io.loadmat(fpath, squeeze_me=False)
        raw = mat['dataLog']

        for row in range(1, raw.shape[0]):
            c = raw[row, 1]
            if c.size == 0:
                continue
            if str(c.flat[0]).strip() == 'Stim on':
                stim_names.append(str(raw[row, 2].flat[0]).strip())

    return stim_names


# beta_groups.csv gives the trial-wise category labels in beta order.
_bl = pd.read_csv(label_file, header=None, names=['beta_file', 'category'])
beta_file_list = _bl['beta_file'].tolist()
template_categories = _bl['category'].tolist()

# Use Sub27 onset files to build a stimulus-name -> category mapping, same as your ROI notebook.
stim_template = load_stim_names_for_subject('Sub27', runs)
if len(stim_template) != len(template_categories):
    raise ValueError(f"Sub27 stimulus count ({len(stim_template)}) != beta label count ({len(template_categories)})")

stim_to_category = {s: c for s, c in zip(stim_template, template_categories)}
print(f"Mapped {len(stim_to_category)} stimulus names to categories.")
print(_bl['category'].value_counts())

Mapped 120 stimulus names to categories.
category
pleasantAI      100
unpleasantAI    100
neutralAI       100
pleasant        100
neutral         100
unpleasant      100
Name: count, dtype: int64


## Mask and explicit KD-tree searchlight grid

In [4]:
# Use a beta image as the reference grid.
temp_beta = os.path.join(beta_dir, 'Sub1', beta_file_list[0])
if not os.path.exists(temp_beta):
    raise FileNotFoundError(temp_beta)

ref_img = nib.load(temp_beta)
mask_img = nib.load(mask_file)

resampled_mask_img = resample_img(
    mask_img,
    target_affine=ref_img.affine,
    target_shape=ref_img.shape,
    interpolation='nearest'
)
mask_bool = resampled_mask_img.get_fdata() > 0
dims = mask_bool.shape
n_vox_total = int(np.prod(dims))

mask_flat_idx = np.flatnonzero(mask_bool.ravel())
n_mask_vox = len(mask_flat_idx)

print(f"Reference beta shape: {dims}")
print(f"Total voxels: {n_vox_total:,}")
print(f"In-mask voxels: {n_mask_vox:,}")

def build_kdtree_searchlight(mask_flat_idx, dims, affine, radius_mm):
    """Return center flat indices and neighbor lists in mask-column coordinates.

    Important:
    - data matrices are loaded as [trial x mask-column].
    - maps are saved as full flattened image vectors.
    Therefore we keep both:
    - centers_flat: flat voxel index in the full image.
    - neighbors_cols: columns into the masked beta matrix.
    """
    ijk = np.column_stack(np.unravel_index(mask_flat_idx, dims))
    xyz = nib.affines.apply_affine(affine, ijk)

    tree = cKDTree(xyz)
    neighbors_cols = tree.query_ball_point(xyz, r=radius_mm)
    neighbors_cols = [np.asarray(n, dtype=np.int32) for n in neighbors_cols]

    centers_flat = mask_flat_idx.astype(np.int64)
    return centers_flat, neighbors_cols

centers_flat, neighbors_cols = build_kdtree_searchlight(
    mask_flat_idx=mask_flat_idx,
    dims=dims,
    affine=resampled_mask_img.affine,
    radius_mm=RADIUS_MM
)

if MAX_CENTERS_DEBUG is not None:
    centers_flat = centers_flat[:MAX_CENTERS_DEBUG]
    neighbors_cols = neighbors_cols[:MAX_CENTERS_DEBUG]

print(f"Searchlight centers: {len(centers_flat):,}")
print(f"Median sphere size: {int(np.median([len(n) for n in neighbors_cols]))} voxels")
print(f"Min / max sphere size: {min(len(n) for n in neighbors_cols)} / {max(len(n) for n in neighbors_cols)}")

Reference beta shape: (79, 95, 79)
Total voxels: 592,895
In-mask voxels: 228,419
Searchlight centers: 228,419
Median sphere size: 81 voxels
Min / max sphere size: 16 / 81


## Z-scoring and decoding helpers

In [5]:
def safe_pattern_zscore(X):
    """Z-score across voxels within each trial/sample, robust to zero variance."""
    X = np.asarray(X, dtype=np.float32)
    Xz = zscore(X, axis=1, nan_policy='omit')
    return np.nan_to_num(Xz, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def fit_trial_zscore(X_train):
    """Fit voxel-wise standardization parameters across stimulus-averaged patterns."""
    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    std[~np.isfinite(std)] = 1.0
    std[std == 0] = 1.0
    return mean.astype(np.float32), std.astype(np.float32)


def apply_trial_zscore(X, mean, std):
    Xz = (X - mean) / std
    return np.nan_to_num(Xz, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def compute_voxel_source_zscore_in_sphere(cond_data_dict):
    """Source-level normalization inside one searchlight sphere.

    Matches the ROI logic (compute_source_zscore / compute_voxel_source_zscore):
    1. pattern normalization across voxels within each (stimulus-averaged) pattern;
    2. voxel-wise standardization across all patterns from the same source
       (Natural source and AI source separately).

    For within-source decoding this is the natural normalization: Natural patterns
    are standardized with Natural statistics, AI with AI statistics.
    """
    result = {}
    for source_cats in [natural_cats, ai_cats]:
        present = [c for c in source_cats if c in cond_data_dict and len(cond_data_dict[c]) > 0]
        if len(present) == 0:
            continue

        all_data = np.vstack([cond_data_dict[c] for c in present]).astype(np.float32)
        sizes = [len(cond_data_dict[c]) for c in present]

        all_z = safe_pattern_zscore(all_data)
        mean = np.mean(all_z, axis=0)
        std = np.std(all_z, axis=0)
        std[~np.isfinite(std)] = 1.0
        std[std == 0] = 1.0
        all_z = ((all_z - mean) / std).astype(np.float32)
        all_z = np.nan_to_num(all_z, nan=0.0, posinf=0.0, neginf=0.0)

        start = 0
        for c, sz in zip(present, sizes):
            result[c] = all_z[start:start+sz]
            start += sz

    return result


def prepare_valid_sphere(cond_data_raw, sphere_cols, min_voxels=MIN_VOXELS):
    """Extract sphere data and remove invalid/constant voxels using all conditions.

    cond_data_raw holds *stimulus-averaged* (ERP) patterns per condition,
    shape [n_unique_stim x n_mask_voxels].
    """
    sphere = {c: cond_data_raw[c][:, sphere_cols] for c in all_categories}

    valid = np.ones(len(sphere_cols), dtype=bool)
    for c in all_categories:
        valid &= np.all(np.isfinite(sphere[c]), axis=0)

    if valid.sum() < min_voxels:
        return None, "too_few_finite_voxels"

    for c in all_categories:
        sphere[c] = sphere[c][:, valid]

    # Remove voxels with no variance across all patterns/conditions in this sphere.
    stacked = np.vstack([sphere[c] for c in all_categories])
    var = np.var(stacked, axis=0)
    valid_var = np.isfinite(var) & (var > 0)

    if valid_var.sum() < min_voxels:
        return None, "too_few_variable_voxels"

    for c in all_categories:
        sphere[c] = sphere[c][:, valid_var].astype(np.float32)

    return sphere, "ok"


def decode_within_erp_searchlight(d1, d2,
                                  n_repeats=ERP_N_REPEATS, n_folds=ERP_N_FOLDS,
                                  use_trial_zscore=USE_TRIAL_ZSCORE_IN_FOLD,
                                  seed_base=42):
    """Within-source ERP (stimulus-averaged) decoding for one sphere.

    d1, d2 are stimulus-averaged patterns for the two classes of ONE source,
    already source-normalized inside the sphere.

    Mirrors decode_within_erp from the ROI notebook: RepeatedStratifiedKFold,
    linear SVM, train and test within the same source. Voxel-wise standardization,
    if enabled, is fit on the training fold only (no leakage).
    """
    n1, n2 = len(d1), len(d2)
    if min(n1, n2) < n_folds:
        return np.nan

    X = np.vstack([d1, d2]).astype(np.float32)
    y = np.array([1] * n1 + [0] * n2)

    if X.shape[1] < MIN_VOXELS:
        return np.nan

    accuracies = []
    for repeat in range(n_repeats):
        skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed_base + repeat)
        for train_idx, test_idx in skf.split(X, y):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            if use_trial_zscore:
                mean, std = fit_trial_zscore(X_train)
                X_train = apply_trial_zscore(X_train, mean, std)
                X_test = apply_trial_zscore(X_test, mean, std)

            if np.all(np.std(X_train, axis=0) == 0):
                continue

            clf = SVC(kernel='linear', C=1.0)
            clf.fit(X_train, y_train)
            accuracies.append(clf.score(X_test, y_test))

    return float(np.mean(accuracies)) if len(accuracies) else np.nan

## Subject loading helper

In [6]:
def make_subject_beta_table(sub):
    stim_names = load_stim_names_for_subject(sub, runs)
    if len(stim_names) != len(beta_file_list):
        raise ValueError(f"{sub}: stimulus count ({len(stim_names)}) != beta count ({len(beta_file_list)})")

    missing = [s for s in stim_names if s not in stim_to_category]
    if len(missing) > 0:
        raise KeyError(f"{sub}: {len(missing)} stimuli not in stim_to_category. First few: {missing[:5]}")

    categories = [stim_to_category[s] for s in stim_names]
    bl = pd.DataFrame({
        'beta_file': beta_file_list,
        'category': categories,
        'stimulus': stim_names,
    })
    return bl


def load_subject_masked_betas(sub, bl):
    """Load beta files into a [n_trials x n_mask_voxels] float32 matrix."""
    sub_dir = os.path.join(beta_dir, sub)
    n_trials = len(bl)

    X = np.empty((n_trials, n_mask_vox), dtype=np.float32)

    for i, beta_file in enumerate(bl['beta_file'].values):
        beta_path = os.path.join(sub_dir, beta_file)
        if not os.path.exists(beta_path):
            if beta_path.endswith('.nii') and os.path.exists(beta_path + '.gz'):
                beta_path = beta_path + '.gz'
            else:
                raise FileNotFoundError(beta_path)

        img = nib.load(beta_path)
        if img.shape != dims:
            raise ValueError(f"{sub} {beta_file}: beta shape {img.shape} does not match mask/reference shape {dims}")

        X[i] = img.get_fdata(dtype=np.float32).ravel()[mask_flat_idx]

    return X


def make_stimulus_averaged_conditions(X, bl):
    """ERP step: average single-trial betas within each stimulus identity.

    Returns a dict cond -> [n_unique_stim x n_mask_voxels], i.e. one pattern per
    unique image, exactly like average_by_stimulus / extract_roi_erp in the ROI notebook.
    """
    cond_data = {}
    for cond in all_categories:
        mask = bl['category'] == cond
        idx_arr = bl.index[mask].to_numpy()
        stim_arr = bl.loc[idx_arr, 'stimulus'].to_numpy()
        cond_raw = X[idx_arr]                      # [n_trials_cond x n_mask_vox]
        unique_stims = np.unique(stim_arr)
        averaged = np.array(
            [np.mean(cond_raw[stim_arr == s], axis=0) for s in unique_stims],
            dtype=np.float32
        )
        cond_data[cond] = averaged
    return cond_data


def print_condition_counts(cond_data, sub):
    print(f"{sub} ERP (stimulus-averaged) pattern counts:")
    for c in all_categories:
        print(f"  {c}: {len(cond_data[c])}")

## Quick diagnostic on one subject before the full run

In [7]:
def run_quick_diagnostic(sub='Sub1', n_centers=500):
    """Run a small subset and report why centers are skipped.

    Use this first if you get all-NaN maps. It should show valid decoding values
    for at least some centers if paths/labels/masks are correct.
    """
    print(f"Diagnostic subject: {sub}")
    bl = make_subject_beta_table(sub)
    X = load_subject_masked_betas(sub, bl)
    cond_raw = make_stimulus_averaged_conditions(X, bl)
    print_condition_counts(cond_raw, sub)

    counters = Counter()
    accs = defaultdict(list)

    n_use = min(n_centers, len(centers_flat))
    for center_i in tqdm(range(n_use), desc=f"diagnostic {sub}"):
        sphere_cols = neighbors_cols[center_i]
        if len(sphere_cols) < MIN_VOXELS:
            counters['too_few_neighbors'] += 1
            continue

        sphere_raw, status = prepare_valid_sphere(cond_raw, sphere_cols)
        counters[status] += 1
        if status != "ok":
            continue

        sphere_z = compute_voxel_source_zscore_in_sphere(sphere_raw)

        for c1, c2 in within_comparisons:
            comp = f"{c1}_vs_{c2}"
            try:
                acc = decode_within_erp_searchlight(
                    sphere_z[c1], sphere_z[c2],
                    n_repeats=3, n_folds=ERP_N_FOLDS
                )
                if np.isfinite(acc):
                    accs[comp].append(acc)
                else:
                    counters['decoder_nan'] += 1
            except Exception as e:
                counters[f'decoder_error:{type(e).__name__}'] += 1

    print("\nSkip/status counters:")
    for k, v in counters.most_common():
        print(f"  {k}: {v}")

    print("\nAccuracies from valid diagnostic centers:")
    for comp in comp_names:
        vals = np.asarray(accs[comp], dtype=float)
        if len(vals) == 0:
            print(f"  {comp}: no finite values")
        else:
            print(f"  {comp}: n={len(vals)}, mean={vals.mean():.3f}, min={vals.min():.3f}, max={vals.max():.3f}")

    del X, cond_raw
    gc.collect()

# Uncomment this before the full run if needed:
# run_quick_diagnostic('Sub1', n_centers=500)

## Main per-subject searchlight

In [8]:
def process_one_subject_searchlight(sub):
    """Run all within-source ERP searchlights for one subject."""
    print(f"\nStarting {sub}")

    bl = make_subject_beta_table(sub)
    X = load_subject_masked_betas(sub, bl)
    cond_raw = make_stimulus_averaged_conditions(X, bl)

    # Full image-length flat maps, with NaN outside valid centers.
    comp_maps = {
        comp: np.full(n_vox_total, np.nan, dtype=np.float32)
        for comp in comp_names
    }

    counters = Counter()

    for center_i in tqdm(range(len(centers_flat)), desc=sub, leave=False):
        center_flat = centers_flat[center_i]
        sphere_cols = neighbors_cols[center_i]

        if len(sphere_cols) < MIN_VOXELS:
            counters['too_few_neighbors'] += 1
            continue

        sphere_raw, status = prepare_valid_sphere(cond_raw, sphere_cols)
        counters[status] += 1
        if status != "ok":
            continue

        # Source normalization is done within the current sphere, per source.
        sphere_z = compute_voxel_source_zscore_in_sphere(sphere_raw)

        for c1, c2 in within_comparisons:
            comp = f"{c1}_vs_{c2}"
            try:
                acc = decode_within_erp_searchlight(
                    sphere_z[c1], sphere_z[c2],
                    n_repeats=ERP_N_REPEATS,
                    n_folds=ERP_N_FOLDS
                )
                if np.isfinite(acc):
                    comp_maps[comp][center_flat] = acc
                else:
                    counters[f'{comp}:decoder_nan'] += 1
            except Exception as e:
                counters[f'{comp}:decoder_error:{type(e).__name__}'] += 1

    # Print quick subject summary.
    print(f"\n{sub} status counters:")
    for k, v in counters.most_common(12):
        print(f"  {k}: {v}")

    for comp, flat_map in comp_maps.items():
        vals = flat_map[np.isfinite(flat_map)]
        if len(vals) == 0:
            print(f"  {sub} | {comp}: ALL NAN")
        else:
            print(f"  {sub} | {comp}: valid={len(vals):,}, mean={vals.mean():.3f}, max={vals.max():.3f}")

    del X, cond_raw
    gc.collect()

    return {'sub': sub, 'maps': comp_maps, 'counters': dict(counters)}

## Run analysis

In [9]:
run_subs = subs[:1] if DEBUG_FIRST_SUB_ONLY else subs

print(f"Running {len(run_subs)} subjects")
print(f"Searchlight radius: {RADIUS_MM} mm")
print(f"Centers: {len(centers_flat):,}")
print(f"Decoding: within-source ERP (stimulus-averaged), repeats={ERP_N_REPEATS}, folds={ERP_N_FOLDS}")
print(f"Parallel jobs: {N_JOBS}")

results_list = Parallel(n_jobs=N_JOBS, prefer='processes', verbose=10)(
    delayed(process_one_subject_searchlight)(sub) for sub in run_subs
)

searchlight_results = {comp: {} for comp in comp_names}
subject_counters = {}

for r in results_list:
    sub = r['sub']
    subject_counters[sub] = r.get('counters', {})
    for comp, flat_map in r['maps'].items():
        searchlight_results[comp][sub] = flat_map.reshape(dims)

print("\nComplete.")

Running 30 subjects
Searchlight radius: 5.0 mm
Centers: 228,419
Decoding: within-source ERP (stimulus-averaged), repeats=10, folds=5
Parallel jobs: 10


[Parallel(n_jobs=10)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=10)]: Done   5 tasks      | elapsed: 807.3min
[Parallel(n_jobs=10)]: Done  15 out of  30 | elapsed: 1907.5min remaining: 1907.5min
[Parallel(n_jobs=10)]: Done  19 out of  30 | elapsed: 1922.4min remaining: 1113.0min
[Parallel(n_jobs=10)]: Done  23 out of  30 | elapsed: 2906.4min remaining: 884.5min
[Parallel(n_jobs=10)]: Done  27 out of  30 | elapsed: 2940.4min remaining: 326.7min
[Parallel(n_jobs=10)]: Done  30 out of  30 | elapsed: 2970.7min finished



Complete.


## Save per-subject maps

In [10]:
sub_output_dir = os.path.join(output_dir, 'per_subject')
os.makedirs(sub_output_dir, exist_ok=True)

for comp in searchlight_results:
    for sub, acc_map_3d in searchlight_results[comp].items():
        nii = nib.Nifti1Image(acc_map_3d.astype(np.float32), resampled_mask_img.affine, resampled_mask_img.header)
        nib.save(nii, os.path.join(sub_output_dir, f"{comp}_{sub}.nii.gz"))

print(f"Saved individual subject maps to: {sub_output_dir}")

Saved individual subject maps to: N:\Experimental_Data\yujunchen\projects\AI_IAPS\Decoding\searchlight_within_source_erp\per_subject


## Average across subjects and save group maps

In [11]:
print("Averaging across subjects and saving NIfTI files...")

summary = {}

for comp, sub_dict in searchlight_results.items():
    sub_maps = [sub_dict[sub] for sub in run_subs if sub in sub_dict]
    if len(sub_maps) == 0:
        print(f"{comp}: no subject maps, skipping.")
        continue

    all_maps = np.stack(sub_maps, axis=0)
    n_valid = np.sum(np.isfinite(all_maps), axis=0).astype(np.float32)

    avg_map = np.full(dims, np.nan, dtype=np.float32)
    std_map = np.full(dims, np.nan, dtype=np.float32)

    valid_vox = n_valid > 0
    avg_map[valid_vox] = np.nanmean(all_maps[:, valid_vox], axis=0)
    std_map[valid_vox] = np.nanstd(all_maps[:, valid_vox], axis=0)

    summary[comp] = {
        'mean': avg_map,
        'std': std_map,
        'n': n_valid,
    }

    nib.save(
        nib.Nifti1Image(avg_map.astype(np.float32), resampled_mask_img.affine, resampled_mask_img.header),
        os.path.join(output_dir, f"{comp}_mean.nii.gz")
    )
    nib.save(
        nib.Nifti1Image(std_map.astype(np.float32), resampled_mask_img.affine, resampled_mask_img.header),
        os.path.join(output_dir, f"{comp}_std.nii.gz")
    )
    nib.save(
        nib.Nifti1Image(n_valid.astype(np.float32), resampled_mask_img.affine, resampled_mask_img.header),
        os.path.join(output_dir, f"{comp}_nsubjects.nii.gz")
    )

    vals = avg_map[np.isfinite(avg_map)]
    print(f"\n{comp}:")
    print(f"  subjects: {len(sub_maps)}")
    print(f"  valid centers: {len(vals):,}")
    if len(vals) > 0:
        print(f"  mean +/- SD: {vals.mean():.3f} +/- {vals.std():.3f}")
        print(f"  max: {vals.max():.3f}")
        print(f"  >0.50: {(vals > 0.50).sum():,}")
        print(f"  >0.55: {(vals > 0.55).sum():,}")

print("\nGroup maps saved.")

Averaging across subjects and saving NIfTI files...

pleasant_vs_neutral:
  subjects: 30
  valid centers: 228,040
  mean +/- SD: 0.561 +/- 0.016
  max: 0.800
  >0.50: 227,831
  >0.55: 173,074

pleasantAI_vs_neutralAI:
  subjects: 30
  valid centers: 228,040
  mean +/- SD: 0.448 +/- 0.020
  max: 0.723
  >0.50: 3,902
  >0.55: 113

unpleasant_vs_neutral:
  subjects: 30
  valid centers: 228,040
  mean +/- SD: 0.487 +/- 0.014
  max: 0.723
  >0.50: 33,943
  >0.55: 298

unpleasantAI_vs_neutralAI:
  subjects: 30
  valid centers: 228,040
  mean +/- SD: 0.446 +/- 0.017
  max: 0.658
  >0.50: 632
  >0.55: 44

Group maps saved.


## Save all results as pickle

In [12]:
all_results = {
    'searchlight_results': searchlight_results,
    'summary': summary,
    'subject_counters': subject_counters,
    'subs': run_subs,
    'dims': dims,
    'mask_flat_idx': mask_flat_idx,
    'centers_flat': centers_flat,
    'within_comparisons': within_comparisons,
    'comp_names': comp_names,
    'analysis': 'within_source_ERP_stimulus_averaged',
    'z_mode': 'voxel_source_zscore_spherewise + inside_fold_trial_zscore',
    'radius_mm': RADIUS_MM,
    'min_voxels': MIN_VOXELS,
    'erp_n_repeats': ERP_N_REPEATS,
    'erp_n_folds': ERP_N_FOLDS,
    'use_trial_zscore_in_fold': USE_TRIAL_ZSCORE_IN_FOLD,
    'notes': {
        'decoding': 'Within-source: train and test on the same source (Natural->Natural, AI->AI).',
        'erp': 'Single trials averaged within each stimulus identity before decoding, matching decode_within_erp.',
        'source_z': 'Source-level z-scoring computed inside each searchlight sphere, per source separately.',
        'fold_z': 'Voxel-wise standardization fit on the training fold only and applied to the test fold (no leakage).',
        'indexing': 'KD-tree returns neighbor lists in mask-column coordinates; map assignment uses full-image flat center indices.',
        'validity': 'Finite and nonzero-variance voxels are checked across all six categories per sphere.',
    }
}

pkl_path = os.path.join(output_dir, 'searchlight_within_source_erp_results.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(all_results, f)

print(f"Saved all results to: {pkl_path}")

Saved all results to: N:\Experimental_Data\yujunchen\projects\AI_IAPS\Decoding\searchlight_within_source_erp\searchlight_within_source_erp_results.pkl


## Peak locations

In [13]:
for comp in summary:
    avg_map = summary[comp]['mean']
    if not np.any(np.isfinite(avg_map)):
        print(f"{comp}: all NaN")
        continue

    flat = avg_map.ravel()
    top_idx = np.nanargmax(flat)
    peak_ijk = np.unravel_index(top_idx, dims)
    peak_acc = flat[top_idx]
    peak_mni = nib.affines.apply_affine(resampled_mask_img.affine, peak_ijk)

    print(f"{comp}:")
    print(f"  peak accuracy: {peak_acc:.3f}")
    print(f"  voxel coords: {peak_ijk}")
    print(f"  MNI coords: ({peak_mni[0]:.1f}, {peak_mni[1]:.1f}, {peak_mni[2]:.1f})")
    print()

pleasant_vs_neutral:
  peak accuracy: 0.800
  voxel coords: (np.int64(30), np.int64(57), np.int64(14))
  MNI coords: (18.0, 2.0, -42.0)

pleasantAI_vs_neutralAI:
  peak accuracy: 0.723
  voxel coords: (np.int64(45), np.int64(55), np.int64(18))
  MNI coords: (-12.0, -2.0, -34.0)

unpleasant_vs_neutral:
  peak accuracy: 0.723
  voxel coords: (np.int64(28), np.int64(54), np.int64(13))
  MNI coords: (22.0, -4.0, -44.0)

unpleasantAI_vs_neutralAI:
  peak accuracy: 0.658
  voxel coords: (np.int64(32), np.int64(55), np.int64(17))
  MNI coords: (14.0, -2.0, -36.0)



## If the output is still all NaN

Run:

```python
run_quick_diagnostic('Sub1', n_centers=500)
```

Then check the counters:
- `too_few_finite_voxels`: mask/beta mismatch, many NaNs in betas, or wrong mask space.
- `too_few_variable_voxels`: sphere has constant signal after masking; lower `MIN_VOXELS` only for testing.
- `decoder_nan`: likely too few stimulus-averaged patterns per class (`min(n1, n2) < ERP_N_FOLDS`).
- no valid diagnostic accuracies: check that `beta_groups.csv`, onset `.mat` files, and beta order match correctly.

For a fast smoke test, set:
```python
DEBUG_FIRST_SUB_ONLY = True
MAX_CENTERS_DEBUG = 2000
ERP_N_REPEATS = 3
N_JOBS = 1
```
Then run the full notebook after confirming finite values.
